In [1]:
# import os

# # Load configuration from YAML file
# config = {
#     "model_name": "llama3-70b-8192",
#     # "model_name": "llama3-8b-8192",
#     # "model_name": "llama-3.1-70b-versatile",
#     # "model_name": "llama-3.1-8b-instant",
#     # "model_name": "gemma2-9b-it",
# }


In [2]:
# Define prompt strings as constants
DESCRIPTION_PROMPT = [
    ("system", """Given the following JSON example(s) for a task type:
     
{raw_example}

Provide a concise description of the task type, including the format and
style of the output. If there are multiple examples, provide a description
for the task type as a whole, ignore the unique parts of the examples.

Format your response as follows:
Task Description: [Your description here]
""")
]

INPUT_ANALYSIS_PROMPT = [
    ("system", """Describe input dimensions, attributes, ranges, and typical values
for a specific task type. Identify main inputs, their impacts, and interactions.
Provide names, descriptions, ranges, and examples for each. Explain how they
affect task execution or results. Include an example of generating comprehensive
input samples using these dimensions and attributes.

Format your response as follows:
Input Analysis: [Your analysis here]
"""),
    ("user", """Task Description:

{description}

""")
]

BRIEFS_PROMPT = [
    ("system", """Given the task type description, and input analysis, generate
descriptions for {generating_batch_size} new examples with detailed attributes
based on this task type. But don't provide any detailed task output.

Use the input analysis to create diverse and comprehensive example briefs that
cover various input dimensions and attribute ranges.

Format your response as a valid YAML object with a single key 'new_example_briefs'
containing a YAML array of {generating_batch_size} objects, each with a
'example_brief' field.
"""),
    ("user", """Task Description:

{description}

Input Analysis:

{input_analysis}

""")
]

EXAMPLES_FROM_BRIEFS_PROMPT = [
    ("system", """Given the task type description, brief descriptions for new examples, 
and JSON example(s), generate {generating_batch_size} more input/output examples for this task type,
strictly based on the brief descriptions. Ensure that the new examples are
consistent with the brief descriptions and do not introduce any new information
not present in the briefs.

Format your response as a valid JSON object with a single key 'examples' 
containing a JSON array of {generating_batch_size} objects, each with 'input' and 'output' fields.
"""),
    ("user", """Task Description:

{description}

New Example Briefs: 

{new_example_briefs}

Example(s):

{raw_example}

""")
]

EXAMPLES_PROMPT = [
    ("system", """Given the task type description, and input/output example(s), generate {generating_batch_size}
new input/output examples for this task type.

Format your response as a valid JSON object with a single key 'examples' 
containing a JSON array of {generating_batch_size} objects, each with 'input' and 'output' fields.
"""),
    ("user", """Task Description:

{description}

Example(s):

{raw_example}

""")
]


In [3]:
import json
import yaml
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import YamlOutputParser


class TaskDescriptionGenerator:
    def __init__(self, model):        
        self.description_prompt = ChatPromptTemplate.from_messages(DESCRIPTION_PROMPT)
        self.input_analysis_prompt = ChatPromptTemplate.from_messages(INPUT_ANALYSIS_PROMPT)
        self.briefs_prompt = ChatPromptTemplate.from_messages(BRIEFS_PROMPT)
        self.examples_from_briefs_prompt = ChatPromptTemplate.from_messages(EXAMPLES_FROM_BRIEFS_PROMPT)
        self.examples_prompt = ChatPromptTemplate.from_messages(EXAMPLES_PROMPT)

        json_model = model.bind(response_format={"type": "json_object"})

        output_parser = StrOutputParser()
        json_parse = JsonOutputParser()

        self.description_chain = self.description_prompt | model | output_parser
        self.input_analysis_chain = self.input_analysis_prompt | model | output_parser
        self.briefs_chain = self.briefs_prompt | model | output_parser
        self.examples_from_briefs_chain = self.examples_from_briefs_prompt | json_model | json_parse
        self.examples_chain = self.examples_prompt | json_model | json_parse

        self.chain = (
            RunnablePassthrough.assign(raw_example = lambda x: json.dumps(x["example"], ensure_ascii=False))
            | RunnablePassthrough.assign(description = self.description_chain)
            | {
                "description": lambda x: x["description"],
                "examples_from_briefs": RunnablePassthrough.assign(input_analysis = lambda x: self.input_analysis_chain.invoke(x))
                    | RunnablePassthrough.assign(new_example_briefs = lambda x: self.briefs_chain.invoke(x)) 
                    | self.examples_from_briefs_chain,
                "examples": self.examples_chain
            }
            | RunnablePassthrough.assign(
                additional_examples=lambda x: (
                    list(x["examples_from_briefs"]["examples"])
                    + list(x["examples"]["examples"])
                )
            )
        )

    def process(self, input_str, generating_batch_size=3):
        try:
            try:
                example_dict = json.loads(input_str)
            except ValueError:
                try:
                    example_dict = yaml.safe_load(input_str)
                except yaml.YAMLError as e:
                    raise ValueError("Invalid input format. Expected a JSON or YAML object.") from e

            # If example_dict is a list, filter out invalid items
            if isinstance(example_dict, list):
                example_dict = [item for item in example_dict if isinstance(item, dict) and 'input' in item and 'output' in item]

            # If example_dict is not a list, check if it's a valid dict
            elif not isinstance(example_dict, dict) or 'input' not in example_dict or 'output' not in example_dict:
                raise ValueError("Invalid input format. Expected an object with 'input' and 'output' fields.")

            # Move the original content to a key named 'example'
            input_dict = {"example": example_dict, "generating_batch_size": generating_batch_size}

            # Invoke the chain with the parsed input dictionary
            result = self.chain.invoke(input_dict)
            return result

        except Exception as e:
            raise RuntimeError(f"An error occurred during processing: {str(e)}")

In [4]:
import gradio as gr

def process_json(input_json, model_name, generating_batch_size=3):
    try:
        model = ChatOpenAI(model=model_name)
        generator = TaskDescriptionGenerator(model)
        result = generator.process(input_json, generating_batch_size)
        description = result["description"]
        examples = [[example["input"], example["output"]] for example in result["additional_examples"]]
        return description, examples
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")

demo = gr.Interface(
    fn=process_json,
    inputs=[
        gr.Textbox(label="Input JSON"),
        gr.Dropdown(label="Model Name", choices=["llama3-70b-8192", "llama3-8b-8192", "llama-3.1-70b-versatile", "llama-3.1-8b-instant", "gemma2-9b-it"], value="llama3-70b-8192"),
        gr.Slider(label="Generating Batch Size", value=3, minimum=1, maximum=10, step=1)
    ],
    outputs=[
        gr.Textbox(label="Description"),
        gr.DataFrame(label="Examples", headers=["Input", "Output"])
    ],
    title="Task Description Generator",
    description="Enter a JSON object with 'input' and 'output' fields to generate a task description and additional examples.",
    allow_flagging="manual",
    flagging_callback=gr.CSVLogger()
)

if __name__ == "__main__":
    demo.launch()

/home/yale/work/meta-prompt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


/home/yale/work/meta-prompt/.venv/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(
